Donations:
Business questions:
1. Which donors are likely to donate again?
2. Which campaigns generate higher lifetime value?
3. Does allocation transparency affect future donations?
4. Does non-monetary engagement lead to later donations?

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths relative to this notebook (ML pipelines/jupyter/ -> sibling lighthouse_csv_files/)
_data_dir = Path("..") / "lighthouse_csv_files"
_donations_csv = _data_dir / "donations.csv"
_donation_allocations_csv = _data_dir / "donation_allocations.csv"
_supporters_csv = _data_dir / "supporters.csv"
donations = pd.read_csv(_donations_csv.resolve())
allocations = pd.read_csv(_donation_allocations_csv.resolve())
supporters = pd.read_csv(_supporters_csv.resolve())

supporters["first_donation_date"] = pd.to_datetime(
    supporters["first_donation_date"], errors="coerce"
)
supporters["created_at"] = pd.to_datetime(supporters["created_at"], errors="coerce")

donations["donation_date"] = pd.to_datetime(donations["donation_date"], errors="coerce")
allocations["allocation_date"] = pd.to_datetime(allocations["allocation_date"], errors="coerce")

donations["donation_year"] = donations["donation_date"].dt.year
donations["donation_month"] = donations["donation_date"].dt.month

# Future: days_since_last_donation — group by supporter_id, sort by donation_date, diff in days vs prior row.

for col in ("amount", "estimated_value"):
    donations[col] = pd.to_numeric(donations[col], errors="coerce")
allocations["amount_allocated"] = pd.to_numeric(allocations["amount_allocated"], errors="coerce")

_donation_value = donations["amount"].where(donations["amount"].notna(), donations["estimated_value"])
donations["donation_value"] = _donation_value
donations["is_zero_value_donation"] = _donation_value.eq(0)
donations["is_negative_value_donation"] = _donation_value.lt(0)

donations.head()

,donation_id,supporter_id,donation_type,donation_date,is_recurring,campaign_name,channel_source,currency_code,amount,estimated_value,impact_unit,notes,referral_post_id,donation_year,donation_month,donation_value,is_zero_value_donation,is_negative_value_donation
0,1,42,Monetary,2025-12-31,False,NaN,Campaign,PHP,717.18,717.18,pesos,In support of safehouse operations,NaN,2025,12,717.18,False,False
1,2,25,Time,2025-12-02,True,Year-End Hope,Event,NaN,NaN,35.15,hours,Community outreach support,NaN,2025,12,35.15,False,False
2,3,19,Monetary,2024-12-02,False,NaN,PartnerReferral,PHP,1074.65,1074.65,pesos,Campaign support,NaN,2024,12,1074.65,False,False
3,4,33,Monetary,2023-09-11,False,NaN,PartnerReferral,PHP,1230.56,1230.56,pesos,In support of safehouse operations,NaN,2023,9,1230.56,False,False
4,5,24,InKind,2023-11-08,True,GivingTuesday,SocialMedia,NaN,NaN,1177.41,items,In support of safehouse operations,421.0,2023,11,1177.41,False,False


In [2]:
donations = donations[
    donations["donation_value"].notna()
    & (donations["donation_value"] > 0)
] # remove zero and negative value donations
donations = donations.dropna(subset=["supporter_id", "donation_date"])
# drop rows with missing supporter_id or donation_date
donations = donations.sort_values(
    ["supporter_id", "donation_date"]
)
# sort by supporter_id and donation_date
donations["next_donation_date"] = donations.groupby("supporter_id")[
    "donation_date"
].shift(-1)
donations["days_to_next_donation"] = (
    donations["next_donation_date"] - donations["donation_date"]
).dt.days
donations["days_since_last_donation"] = (
    donations["donation_date"] - donations.groupby("supporter_id")["donation_date"].shift(1)
).dt.days


In [3]:
# Allocation-level features (one row per donation_id)
_by_area = (
    allocations.groupby(["donation_id", "program_area"], as_index=False)["amount_allocated"]
    .sum()
)
_total_allocated = allocations.groupby("donation_id")["amount_allocated"].sum().rename("total_amount_allocated")
_num_alloc = allocations.groupby("donation_id").size().rename("num_allocations")

_area_with_total = _by_area.merge(_total_allocated.reset_index(), on="donation_id", validate="many_to_one")
_area_with_total["share_of_allocated"] = _area_with_total["amount_allocated"] / _area_with_total[
    "total_amount_allocated"
].replace(0, np.nan)
_concentration = _area_with_total.groupby("donation_id")["share_of_allocated"].max().rename(
    "allocation_concentration"
)

alloc_features = (
    pd.concat([_num_alloc, _total_allocated, _concentration], axis=1)
    .reset_index()
    .merge(donations[["donation_id", "donation_value"]], on="donation_id", how="left", validate="one_to_one")
)

# Share of donation_value covered by allocations (1.0 == 100%); NaN if donation_value is 0 or missing
alloc_features["pct_donation_allocated"] = alloc_features["total_amount_allocated"] / alloc_features[
    "donation_value"
].replace(0, np.nan)

# True when summed allocations match donation_value within tolerance (fully allocated / "sums to 100%")
_alloc_rtol = 1e-5
_alloc_atol = 0.01
_alloc_tot = alloc_features["total_amount_allocated"]
_dv = alloc_features["donation_value"]
alloc_features["is_allocation_fully_specified"] = (
    _dv.notna()
    & _dv.gt(0)
    & np.isclose(_alloc_tot, _dv, rtol=_alloc_rtol, atol=_alloc_atol)
)

alloc_features.head()

,donation_id,num_allocations,total_amount_allocated,allocation_concentration,donation_value,pct_donation_allocated,is_allocation_fully_specified
0,1,1,717.18,1.0,717.18,1.000000,True
1,2,1,35.15,1.0,35.15,1.000000,True
2,3,1,1074.65,1.0,1074.65,1.000000,True
3,4,1,799.86,1.0,1230.56,0.649997,False
4,5,1,1177.41,1.0,1177.41,1.000000,True


In [4]:
### Model base (`model_df`)

# One row per `donation_id`: donations + allocation features + supporter attributes (no email/phone/name PII for modeling).

In [5]:
_alloc_merge = alloc_features.drop(columns=["donation_value"], errors="ignore")

model_df = donations.merge(
    _alloc_merge,
    on="donation_id",
    how="left",
    validate="one_to_one",
)

_supporter_for_model = supporters.drop(
    columns=[
        "display_name",
        "organization_name",
        "first_name",
        "last_name",
        "email",
        "phone",
    ],
    errors="ignore",
)

model_df = model_df.merge(
    _supporter_for_model,
    on="supporter_id",
    how="left",
    validate="many_to_one",
)

model_df = model_df.sort_values(["supporter_id", "donation_date"]).reset_index(drop=True)

print(
    "Row counts — donations (filtered):",
    len(donations),
    "| alloc_features:",
    len(alloc_features),
    "| supporters:",
    len(supporters),
    "| model_df:",
    len(model_df),
)

model_df["is_non_monetary"] = model_df["donation_type"].ne("Monetary")
_nm_cum = model_df.groupby("supporter_id")["is_non_monetary"].cumsum()
model_df["prior_non_monetary_count"] = (
    _nm_cum - model_df["is_non_monetary"].astype(int)
).clip(lower=0)
model_df["prior_donation_index"] = model_df.groupby("supporter_id").cumcount()
model_df["next_donation_type"] = model_df.groupby("supporter_id")["donation_type"].shift(-1)

model_df.head(3)

Row counts — donations (filtered): 420 | alloc_features: 420 | supporters: 60 | model_df: 420


,donation_id,supporter_id,donation_type,donation_date,is_recurring,campaign_name,channel_source,currency_code,amount,estimated_value,...,region,country,status,created_at,first_donation_date,acquisition_channel,is_non_monetary,prior_non_monetary_count,prior_donation_index,next_donation_type
0,145,1,Monetary,2023-03-25,True,NaN,SocialMedia,PHP,774.61,774.61,...,Luzon,Philippines,Active,2022-01-01,2023-07-02,SocialMedia,False,0,0,InKind
1,158,1,InKind,2023-06-24,True,Back to School,Direct,NaN,NaN,606.91,...,Luzon,Philippines,Active,2022-01-01,2023-07-02,SocialMedia,True,0,1,Monetary
2,295,1,Monetary,2023-07-01,True,NaN,Event,PHP,663.94,663.94,...,Luzon,Philippines,Active,2022-01-01,2023-07-02,SocialMedia,False,1,2,InKind


### Outcomes (prediction horizon)

Use **H = 180** days. **`repeat_within_h`**: 1 if the supporter’s **next** donation is within H days; 0 if we observe H days from `donation_date` to `dataset_end` with **no** further donation; **NaN** if censored (no next row and follow-up &lt; H). Models use only **eligible** rows (not censored).


In [6]:
H = 180
_dataset_end = model_df["donation_date"].max()

_has_next = model_df["next_donation_date"].notna()
_window_days = (_dataset_end - model_df["donation_date"]).dt.days

model_df["repeat_within_h_eligible"] = _has_next | (_window_days >= H)

model_df["repeat_within_h"] = np.nan
model_df.loc[_has_next, "repeat_within_h"] = (
    model_df.loc[_has_next, "days_to_next_donation"] <= H
).astype(float)
model_df.loc[~_has_next & (_window_days >= H), "repeat_within_h"] = 0.0

# Q4-related: next gift within H is monetary (NaN if censored / no next in window)
model_df["next_monetary_within_h"] = np.nan
_m_next = _has_next & (model_df["days_to_next_donation"] <= H)
model_df.loc[_m_next, "next_monetary_within_h"] = model_df.loc[
    _m_next, "next_donation_type"
].eq("Monetary").astype(float)
model_df.loc[~_has_next & (_window_days >= H), "next_monetary_within_h"] = 0.0
model_df["next_monetary_eligible"] = model_df["repeat_within_h_eligible"]

print("dataset_end", _dataset_end.date())
print(
    "repeat_within_h eligible:",
    int(model_df["repeat_within_h_eligible"].sum()),
    "| labeled:",
    int(model_df["repeat_within_h"].notna().sum()),
    "| positive rate (labeled):",
    f"{model_df.loc[model_df['repeat_within_h'].notna(), 'repeat_within_h'].mean():.3f}",
)

dataset_end 2026-03-01
repeat_within_h eligible: 382 | labeled: 382 | positive rate (labeled): 0.715


### Q1 — Which donors are likely to donate again?

**Target:** `repeat_within_h` (binary). **Validation:** time-aware **expanding-window** CV (`TimeSeriesSplit`): within each fold, no training row is dated **after** any test row—aligned with training on history and evaluating on a future slice. Same supporter may still appear in both sides; interpret as associational, not causal. **Model:** balanced logistic regression on donation- and supporter-level features. After CV, the pipeline is **refit** on the full time-ordered labeled set for inspection or reuse.

In [16]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

_numeric_for_q1 = [
    "donation_value",
    "days_since_last_donation",
    "prior_non_monetary_count",
    "prior_donation_index",
    "num_allocations",
    "pct_donation_allocated",
    "allocation_concentration",
    "is_recurring",
]
_categorical_for_q1 = [
    "donation_type",
    "channel_source",
    "campaign_name",
    "supporter_type",
    "region",
    "country",
    "relationship_type",
    "status",
    "acquisition_channel",
]

labeled_q1 = model_df[
    model_df["repeat_within_h_eligible"] & model_df["repeat_within_h"].notna()
].copy()
labeled_q1_sorted = labeled_q1.sort_values(
    ["donation_date", "donation_id"], kind="mergesort"
).reset_index(drop=True)

X_q1 = labeled_q1_sorted[_numeric_for_q1 + _categorical_for_q1].copy()
for _col in _numeric_for_q1:
    X_q1[_col] = pd.to_numeric(X_q1[_col], errors="coerce")
X_q1[_categorical_for_q1] = X_q1[_categorical_for_q1].fillna("Unknown")
y_q1 = labeled_q1_sorted["repeat_within_h"].astype(int)

_prep_q1 = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            _numeric_for_q1,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "oh",
                        OneHotEncoder(handle_unknown="ignore", max_categories=30),
                    ),
                ]
            ),
            _categorical_for_q1,
        ),
    ]
)
_q1_base = Pipeline(
    steps=[
        ("prep", _prep_q1),
        (
            "lr",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                solver="saga",
                random_state=42,
            ),
        ),
    ]
)

# TimeSeriesSplit respects row order — early folds have small train sets (tradeoff for ~n rows).
_n_splits_q1 = 4
_tscv_q1 = TimeSeriesSplit(n_splits=_n_splits_q1)
_fold_aucs = []
_fold_pr_aucs = []
for _fold_i, (_train_idx, _test_idx) in enumerate(_tscv_q1.split(X_q1), start=1):
    _est = clone(_q1_base)
    _X_tr, _X_te = X_q1.iloc[_train_idx], X_q1.iloc[_test_idx]
    _y_tr, _y_te = y_q1.iloc[_train_idx], y_q1.iloc[_test_idx]
    _est.fit(_X_tr, _y_tr)
    _proba_te = _est.predict_proba(_X_te)[:, 1]
    if _y_te.nunique() < 2:
        print(
            f"Q1 CV fold {_fold_i} skipped (single class in test block, n={len(_y_te)})"
        )
        continue
    _auc = roc_auc_score(_y_te, _proba_te)
    _pr_auc = average_precision_score(_y_te, _proba_te)
    _fold_aucs.append(_auc)
    _fold_pr_aucs.append(_pr_auc)
    print(
        "Q1 CV fold",
        _fold_i,
        "| train",
        len(_train_idx),
        "test",
        len(_test_idx),
        "| ROC-AUC",
        f"{_auc:.3f}",
        "PR-AUC",
        f"{_pr_auc:.3f}",
    )

if _fold_aucs:
    print(
        "Q1 CV mean ROC-AUC:",
        f"{np.mean(_fold_aucs):.3f} (+/- {np.std(_fold_aucs):.3f})",
        "| mean PR-AUC:",
        f"{np.mean(_fold_pr_aucs):.3f} (+/- {np.std(_fold_pr_aucs):.3f})",
    )

_q1_clf = clone(_q1_base)
_q1_clf.fit(X_q1, y_q1)
print("Q1 refit on full time-ordered labeled set, n =", len(X_q1))

Q1 CV fold 1 | train 78 test 76 | ROC-AUC 0.632 PR-AUC 0.696
Q1 CV fold 2 | train 154 test 76 | ROC-AUC 0.678 PR-AUC 0.793
Q1 CV fold 3 | train 230 test 76 | ROC-AUC 0.620 PR-AUC 0.811
Q1 CV fold 4 | train 306 test 76 | ROC-AUC 0.576 PR-AUC 0.867
Q1 CV mean ROC-AUC: 0.627 (+/- 0.036) | mean PR-AUC: 0.792 (+/- 0.062)
Q1 refit on full time-ordered labeled set, n = 382


### Q2 — Which campaigns drive higher lifetime value (LTV)?

**LTV** = sum of `donation_value` per `supporter_id` in this extract. **First-touch** campaign = `campaign_name` on the supporter’s earliest donation (missing → `Unknown`). Table is descriptive; R² is in-sample only from a simple linear regression on one-hot campaigns (use for storytelling, not hold-out quality).

In [8]:
_first_touch = (
    model_df.sort_values("donation_date")
    .groupby("supporter_id", as_index=False)
    .first()[["supporter_id", "campaign_name"]]
)
_first_touch["first_campaign"] = _first_touch["campaign_name"].fillna("Unknown")
_ltv_by_supporter = (
    model_df.groupby("supporter_id", as_index=False)["donation_value"]
    .sum()
    .rename(columns={"donation_value": "ltv"})
)
_ltv_campaign = _first_touch.merge(
    _ltv_by_supporter, on="supporter_id", how="inner", validate="one_to_one"
)

ltv_by_first_campaign = (
    _ltv_campaign.groupby("first_campaign")["ltv"]
    .agg(n_supporters="count", mean_ltv="mean", median_ltv="median")
    .sort_values("mean_ltv", ascending=False)
)
ltv_by_first_campaign

X_q2 = _ltv_campaign[["first_campaign"]]
y_q2 = _ltv_campaign["ltv"]
_prep_q2 = ColumnTransformer(
    [("c", OneHotEncoder(handle_unknown="ignore"), ["first_campaign"])]
)
_reg_ltv = Pipeline([("prep", _prep_q2), ("lr", LinearRegression())])
_reg_ltv.fit(X_q2, y_q2)
print(
    "Q2 in-sample R² (LTV ~ first-touch campaign):",
    f"{_reg_ltv.score(X_q2, y_q2):.3f}",
)

Q2 in-sample R² (LTV ~ first-touch campaign): 0.046


### Q3 — Does allocation “transparency” associate with repeat giving?

**Proxy features:** allocation counts, `%` allocated to programs, concentration, `is_allocation_fully_specified`. Same target as Q1 (`repeat_within_h`). **Split:** single temporal hold-out at the 70th percentile of `donation_date` (unlike Q1’s expanding-window CV). **Interpretation:** association only; allocation is observed with the donation, not a randomized intervention.

In [9]:
_transparency_features = [
    "num_allocations",
    "pct_donation_allocated",
    "allocation_concentration",
    "is_allocation_fully_specified",
]

labeled_q3 = model_df[
    model_df["repeat_within_h_eligible"] & model_df["repeat_within_h"].notna()
].copy()
labeled_q3["_alloc_full_f"] = (
    labeled_q3["is_allocation_fully_specified"].fillna(False).astype(float)
)

_time_cut_q3 = labeled_q3["donation_date"].quantile(0.7)
_train_q3 = labeled_q3[labeled_q3["donation_date"] < _time_cut_q3]
_test_q3 = labeled_q3[labeled_q3["donation_date"] >= _time_cut_q3]

_use_q3 = [c for c in _transparency_features if c != "is_allocation_fully_specified"] + [
    "_alloc_full_f"
]
X_train_q3 = _train_q3[_use_q3].apply(pd.to_numeric, errors="coerce")
X_test_q3 = _test_q3[_use_q3].apply(pd.to_numeric, errors="coerce")

y_train_q3 = _train_q3["repeat_within_h"].astype(int)
y_test_q3 = _test_q3["repeat_within_h"].astype(int)

_prep_q3 = Pipeline(
    [
        ("imp", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
_q3_clf = Pipeline(
    steps=[
        ("prep", _prep_q3),
        (
            "lr",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                solver="saga",
                random_state=42,
            ),
        ),
    ]
)
_q3_clf.fit(X_train_q3, y_train_q3)
_proba_q3 = _q3_clf.predict_proba(X_test_q3)[:, 1]
print(
    "Q3 transparency-only ROC-AUC:",
    f"{roc_auc_score(y_test_q3, _proba_q3):.3f}",
)
print(
    "PR-AUC:",
    f"{average_precision_score(y_test_q3, _proba_q3):.3f}",
)

Q3 transparency-only ROC-AUC: 0.486
PR-AUC: 0.759


### Q4 — Does non-monetary engagement relate to a later **monetary** gift?

**Target:** `next_monetary_within_h` — among donations with a **known** next gift within H days, 1 if that next gift is `Monetary`; if no next gift but the supporter is observed for **H** days after the donation, 0. **Features** include `donation_type`, prior non-monetary count, etc. **Split:** 70th percentile date cut (same style as Q3, not Q1’s `TimeSeriesSplit`).

In [10]:
_numeric_for_q4 = [
    "donation_value",
    "days_since_last_donation",
    "prior_non_monetary_count",
    "prior_donation_index",
    "num_allocations",
    "is_recurring",
]
_categorical_for_q4 = [
    "donation_type",
    "channel_source",
    "impact_unit",
    "supporter_type",
]

labeled_q4 = model_df[
    model_df["next_monetary_eligible"] & model_df["next_monetary_within_h"].notna()
].copy()
_time_cut_q4 = labeled_q4["donation_date"].quantile(0.7)
_train_q4 = labeled_q4[labeled_q4["donation_date"] < _time_cut_q4]
_test_q4 = labeled_q4[labeled_q4["donation_date"] >= _time_cut_q4]

X_train_q4 = _train_q4[_numeric_for_q4 + _categorical_for_q4].copy()
X_test_q4 = _test_q4[_numeric_for_q4 + _categorical_for_q4].copy()
for _col in _numeric_for_q4:
    X_train_q4[_col] = pd.to_numeric(X_train_q4[_col], errors="coerce")
    X_test_q4[_col] = pd.to_numeric(X_test_q4[_col], errors="coerce")
X_train_q4[_categorical_for_q4] = X_train_q4[_categorical_for_q4].fillna("Unknown")
X_test_q4[_categorical_for_q4] = X_test_q4[_categorical_for_q4].fillna("Unknown")

y_train_q4 = _train_q4["next_monetary_within_h"].astype(int)
y_test_q4 = _test_q4["next_monetary_within_h"].astype(int)

print(
    "Q4 train / test rows:",
    len(_train_q4),
    len(_test_q4),
    "| positive rate (next within H is Monetary) test:",
    f"{y_test_q4.mean():.3f}",
)

_prep_q4 = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            _numeric_for_q4,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "oh",
                        OneHotEncoder(handle_unknown="ignore", max_categories=25),
                    ),
                ]
            ),
            _categorical_for_q4,
        ),
    ]
)
_q4_clf = Pipeline(
    steps=[
        ("prep", _prep_q4),
        (
            "lr",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                solver="saga",
                random_state=42,
            ),
        ),
    ]
)
_q4_clf.fit(X_train_q4, y_train_q4)
_proba_q4 = _q4_clf.predict_proba(X_test_q4)[:, 1]
print("ROC-AUC:", f"{roc_auc_score(y_test_q4, _proba_q4):.3f}")
print("PR-AUC:", f"{average_precision_score(y_test_q4, _proba_q4):.3f}")
print(classification_report(y_test_q4, (_proba_q4 >= 0.5).astype(int), digits=3))

Q4 train / test rows: 206 88 | positive rate (next within H is Monetary) test: 0.511
ROC-AUC: 0.434
PR-AUC: 0.490
              precision    recall  f1-score   support

           0      0.440     0.256     0.324        43
           1      0.492     0.689     0.574        45

    accuracy                          0.477        88
   macro avg      0.466     0.472     0.449        88
weighted avg      0.467     0.477     0.452        88



### Summary

- **~400 donations** → keep models simple; read PR-AUC when positives are rare; compare to the majority baseline.
- **Censoring:** rows with no next donation and &lt; H days of follow-up are excluded from supervised labels (`repeat_within_h`, `next_monetary_within_h` as NaN).
- **Leakage:** features use only current/prior information on the donation row (plus merged static supporter fields). Do not use future outcomes as inputs.
- **Causality:** Q3–Q4 report **associations** with business proxies, not causal effects of transparency or volunteer behavior.